# IMPORTS

In [2]:
# ==============================
# NLP и текстовая обработка
# ==============================
import re
import string

import nltk
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer

from razdel import tokenize, sentenize
import pymorphy3  # для русского языка
import spacy

# ==============================
# Работа с данными и векторами
# ==============================
import numpy as np
from numpy.linalg import norm

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

# ==============================
# Модели и обучение
# ==============================
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn import svm
from sklearn.linear_model import SGDClassifier, LogisticRegression

# ==============================
# Векторные представления слов
# ==============================
import gensim.downloader as api
from gensim.models import Word2Vec

## NLP Practice

### Tokenisation

In [3]:
# @title Default title text
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [4]:
data = "All work and no play makes jack a dull boy, all work and no play"
tokens = word_tokenize(data.lower())  #uses TreeBankWordTokenizer
print(tokens)

['all', 'work', 'and', 'no', 'play', 'makes', 'jack', 'a', 'dull', 'boy', ',', 'all', 'work', 'and', 'no', 'play']


https://www.nltk.org/api/nltk.tokenize.treebank.html#nltk.tokenize.treebank.TreebankWordTokenizer

In [5]:
print(sent_tokenize("I was going home when she rung. It was a surprise."))

['I was going home when she rung.', 'It was a surprise.']


[<img src="https://raw.githubusercontent.com/natasha/natasha-logos/master/natasha.svg">](https://github.com/natasha/natasha)

[Razdel](https://natasha.github.io/razdel/)

In [6]:
text = 'Кружка-термос на 0.5л (50/64 см³, 516;...)'
list(tokenize(text))

[Substring(0, 13, 'Кружка-термос'),
 Substring(14, 16, 'на'),
 Substring(17, 20, '0.5'),
 Substring(20, 21, 'л'),
 Substring(22, 23, '('),
 Substring(23, 28, '50/64'),
 Substring(29, 32, 'см³'),
 Substring(32, 33, ','),
 Substring(34, 37, '516'),
 Substring(37, 38, ';'),
 Substring(38, 41, '...'),
 Substring(41, 42, ')')]

#### RegEx

Everything on regex https://habr.com/ru/post/349860/

In [7]:
word = 'supercalifragilisticexpialidocious'
re.findall('[abc]|up|super', word)

['super', 'c', 'a', 'a', 'c', 'a', 'c']

In [8]:
re.findall('\d{1,3}', 'These are some numbers: 49 and 432312')

['49', '432', '312']

In [9]:
re.sub('[,\.?!]', '', 'How, to? split. text!')

'How to split text'

In [10]:
re.sub('[^A-z]', ' ', 'I 123 can 45 play 67 football').split()

['I', 'can', 'play', 'football']

### Deleting uninformative words

#### N-gramms

In [11]:
unigram = list(nltk.ngrams(tokens, 1))
bigram = list(nltk.ngrams(tokens, 2))
print(unigram[:5])
print(bigram[:5])

[('all',), ('work',), ('and',), ('no',), ('play',)]
[('all', 'work'), ('work', 'and'), ('and', 'no'), ('no', 'play'), ('play', 'makes')]


In [12]:

print('Popular unigramms: ', FreqDist(unigram).most_common(5))
print('Popular bigramms: ', FreqDist(bigram).most_common(5))

Popular unigramms:  [(('all',), 2), (('work',), 2), (('and',), 2), (('no',), 2), (('play',), 2)]
Popular bigramms:  [(('all', 'work'), 2), (('work', 'and'), 2), (('and', 'no'), 2), (('no', 'play'), 2), (('play', 'makes'), 1)]


#### Stopwords

In [13]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [14]:
stopWords = set(stopwords.words('english'))
print(stopWords)

{'can', 'have', 're', 'i', 's', "they'd", "it'll", "we're", 'itself', "you'd", 'isn', 'him', 'over', "they'll", 'up', "hadn't", 'so', "isn't", "i'd", 'themselves', "shan't", 'out', 'after', 'shouldn', 'both', "needn't", 'me', 'such', 'than', 'because', "he'll", "you'll", 'he', 'she', 'then', "weren't", 'yours', "couldn't", 'o', 'to', 'has', 'does', 'under', 'an', 'his', 'in', 'her', "she'd", 'not', 'their', 'other', 'ours', 'which', 'your', 'any', "doesn't", 'that', 'm', 'from', "i've", 'are', "don't", 'during', 'on', 'below', 'wouldn', 'by', "wasn't", 'won', 'doesn', "hasn't", 'there', 'if', 've', 'my', 'you', 'them', "she'll", 'some', "that'll", 'we', "he'd", 'before', 'down', 'been', "we've", 'who', 'a', "shouldn't", 'until', 'the', 'here', 'our', 'as', 'himself', 'about', 'hers', 'how', 'll', "they've", 'ma', 'had', 'off', 'above', "mightn't", "they're", 'what', "won't", 'was', 'they', 'once', 'should', "it'd", 'again', "she's", 'hasn', 'now', 'at', "it's", 'against', 'further', 't

In [15]:
print([word for word in tokens if word not in stopWords])

['work', 'play', 'makes', 'jack', 'dull', 'boy', ',', 'work', 'play']


In [16]:
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


#### Stemming and Lemmatization
* ‘Caring’ -> Lemmatization -> ‘Care’
* ‘Caring’ -> Stemming -> ‘Car’

### Stemming
* finding the base form of a word

In [17]:
words = ["game", "gaming", "gamed", "games", "compacted"]

In [18]:
ps = PorterStemmer()
list(map(ps.stem, words))

['game', 'game', 'game', 'game', 'compact']

### Lemmatization
* finding the vocabulary from of a word

In [19]:
raw = """DENNIS: Listen, strange women lying in ponds distributing swords
is no basis for a system of government.  Supreme executive power derives from
a mandate from the masses, not from some farcical aquatic ceremony."""

raw_ru = """Не существует научных доказательств в пользу эффективности НЛП, оно
признано псевдонаукой. Систематические обзоры указывают, что НЛП основано на
устаревших представлениях об устройстве мозга, несовместимо с современной
неврологией и содержит ряд фактических ошибок."""

In [20]:
morph = pymorphy3.MorphAnalyzer()
pymorphy_results = list(map(lambda x: morph.parse(x), raw_ru.split(' ')))
print(' '.join([res[0].normal_form for res in pymorphy_results]))

не существовать научный доказательство в польза эффективность нлп, оно
признать псевдонаукой. систематический обзор указывают, что нлп основать на
устаревший представление о устройство мозга, несовместимый с современной
неврология и содержать ряд фактический ошибок.


In [21]:
# 2
nlp = spacy.load('en_core_web_sm')
spacy_results = nlp(raw)
print(' '.join([token.lemma_ for token in spacy_results]))

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

### Part-of-Speech

In [ ]:
# 1
[(res[0].normal_form, res[0].tag) for res in pymorphy_results[:9]]

In [ ]:
# 2
[(token.lemma_, token.pos_) for token in spacy_results[:7]]

### Named entities recognition

In [ ]:
doc = nlp('Apple is looking at buying U.K. startup for $1 billion')

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

### Classification

#### 20 newsgroups
18000 news, 20 groups

In [ ]:
newsgroups_train = fetch_20newsgroups(subset='train')

In [ ]:
newsgroups_train.target_names

In [ ]:
newsgroups_train.filenames.shape

#### Sample

In [ ]:
categories = ['alt.atheism', 'talk.religion.misc',
              'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(subset='train',
                                      categories=categories)
newsgroups_train.filenames.shape

In [ ]:
print(newsgroups_train.data[0])

In [ ]:
newsgroups_train.target[:10]

#### TF-IDF

#### Arguments:
* input : string {‘filename’, ‘file’, ‘content’}
*  lowercase : boolean, default True
*  preprocessor : callable or None (default)
*  tokenizer : callable or None (default)
*  stop_words : string {‘english’}, list, or None (default)
*  ngram_range : tuple (min_n, max_n)
*  max_df : float in range [0.0, 1.0] or int, default=1.0
*  min_df : float in range [0.0, 1.0] or int, default=1
*  max_features : int or None, default=None

In [ ]:
# lowercase
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

In [ ]:
vectorizer = TfidfVectorizer(lowercase=False)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

In [ ]:
vectorizer.get_feature_names_out()[:10]

In [ ]:
# min_df, max_df
vectorizer = TfidfVectorizer(min_df=0.8)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

In [ ]:
vectorizer.get_feature_names_out()

In [ ]:
vectorizer = TfidfVectorizer(min_df=0.01, max_df=0.8)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

In [ ]:
# ngram_range
vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=0.03, max_df=0.9)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

In [ ]:
stopWords = set(stopwords.words('english'))
nltk.download('wordnet')
wnl = nltk.WordNetLemmatizer()


def preproc_nltk(text):
    #text = re.sub(f'[{string.punctuation}]', ' ', text)
    return ' '.join([wnl.lemmatize(word) for word in word_tokenize(text.lower()) if word not in stopWords])


st = "Oh, I think I ve landed Where there are miracles at work,  For the thirst and for the hunger Come the conference of birds"
preproc_nltk(st)

In [ ]:
%%time
vectorizer = TfidfVectorizer(preprocessor=preproc_nltk)
vectors = vectorizer.fit_transform(newsgroups_train.data)

In [ ]:
# preproc_spacy
nlp = spacy.load("en_core_web_sm")
texts = newsgroups_train.data.copy()


def preproc_spacy(text):
    spacy_results = nlp(text)
    return ' '.join([token.lemma_ for token in spacy_results if token.lemma_ not in stopWords])


preproc_spacy(st)

In [ ]:
%%time
new_texts = []
for doc in nlp.pipe(texts, batch_size=32, n_process=3, disable=["parser", "ner"]):
    new_texts.append(' '.join([tok.lemma_ for tok in doc if tok.lemma not in stopWords]))
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(new_texts)

#### Final Model

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_df=0.5, max_features=1000)
vectors = vectorizer.fit_transform(new_texts)
vectorizer.get_feature_names_out()[::100]

#### Cosine similarity

In [ ]:
type(vectors)

In [ ]:
vector = vectors.todense()[0]
(vector != 0).sum()

In [ ]:
vector.shape

In [ ]:
np.mean(list(map(lambda x: (x != 0).sum(), vectors.todense())))

In [ ]:
dense_vectors = vectors.todense()
dense_vectors.shape

In [ ]:
def cosine_sim(v1, v2):
    # v1, v2 (1 x dim)
    return np.array(v1 @ v2.T / norm(v1) / norm(v2))[0][0]

In [ ]:
cosine_sim(dense_vectors[0], dense_vectors[0])

In [ ]:
cosines = []
for i in range(10):
    cosines.append(cosine_sim(dense_vectors[0], dense_vectors[i]))

In [ ]:
# [1, 3, 2, 0, 2, 0, 2, 1, 2, 1]
cosines

#### Training a model on features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(dense_vectors, newsgroups_train.target, test_size=0.2,
                                                    random_state=0)
y_train.shape, y_test.shape

In [ ]:
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)
X_test = np.asarray(X_test)
y_test = np.asarray(y_test)

In [ ]:
sgd = SGDClassifier()
sgd.fit(X_train, y_train)
accuracy_score(y_test, sgd.predict(X_test))

### Embeddings

In [ ]:
embeddings_pretrained = api.load('glove-twitter-25')

In [ ]:
proc_words = [preproc_nltk(text).split() for text in newsgroups_train.data]
embeddings_trained = Word2Vec(proc_words,  # data for model to train on
                              vector_size=100,  # embedding vector size
                              min_count=3,  # consider words that occured at least 5 times
                              window=3).wv

In [ ]:
def vectorize_sum(comment, embeddings):
    """
    implement a function that converts preprocessed comment to a sum of token vectors
    """
    embedding_dim = embeddings.vectors.shape[1]
    features = np.zeros([embedding_dim], dtype='float32')

    for word in preproc_nltk(comment).split():
        if word in embeddings:
            features += embeddings[f'{word}']

    return features

In [ ]:
len(embeddings_trained.index_to_key)

In [ ]:
X_wv = np.stack([vectorize_sum(text, embeddings_pretrained) for text in newsgroups_train.data])
X_train_wv, X_test_wv, y_train, y_test = train_test_split(X_wv, newsgroups_train.target, test_size=0.2, random_state=0)
X_train_wv.shape, X_test_wv.shape

In [ ]:
clf = LogisticRegression(max_iter=5000)
wv_model = clf.fit(X_train_wv, y_train)
accuracy_score(y_test, wv_model.predict(X_test_wv))

In [ ]:
X_wv = np.stack([vectorize_sum(text, embeddings_trained) for text in newsgroups_train.data])
X_train_wv, X_test_wv, y_train, y_test = train_test_split(X_wv, newsgroups_train.target, test_size=0.2, random_state=0)
X_train_wv.shape, X_test_wv.shape

In [ ]:
clf = LogisticRegression(max_iter=10000)
wv_model = clf.fit(X_train_wv, y_train)
accuracy_score(y_test, wv_model.predict(X_test_wv))